# PURITY Validation

Validates row alignment and computes basic event-level accuracy from inference output.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

from pioneerml.integration.zenml import utils as zenml_utils

PROJECT_ROOT = Path(zenml_utils.find_project_root()).resolve()

## Load Source + Predictions

In [ ]:
source_path = PROJECT_ROOT / 'artifacts' / 'purity_notebook_inference.parquet'
pred_path = PROJECT_ROOT / 'artifacts' / 'purity_notebook_predictions' / 'purity_notebook_inference_preds.parquet'

source_tbl = pq.read_table(source_path, columns=['event_id', 'truth_is_signal'])
pred_tbl = pq.read_table(pred_path, columns=['event_id', 'pred_purity_signal'])

source_event = np.asarray(source_tbl.column('event_id').to_numpy(zero_copy_only=False), dtype=np.int64)
source_truth = np.asarray(source_tbl.column('truth_is_signal').to_numpy(zero_copy_only=False), dtype=np.int64)
truth_by_event = {int(e): int(y) for e, y in zip(source_event, source_truth, strict=True)}

pred_event = np.asarray(pred_tbl.column('event_id').to_numpy(zero_copy_only=False), dtype=np.int64)
pred_lists = pred_tbl.column('pred_purity_signal').to_pylist()
pred_prob = np.asarray([float(v[0]) if isinstance(v, list) and len(v) > 0 else np.nan for v in pred_lists], dtype=np.float32)

pred_true = np.asarray([truth_by_event[int(e)] for e in pred_event], dtype=np.int64)
pred_hat = (pred_prob >= 0.5).astype(np.int64)
acc = float((pred_hat == pred_true).mean()) if pred_true.size > 0 else float('nan')

print('rows(source):', source_tbl.num_rows)
print('rows(pred):', pred_tbl.num_rows)
print('accuracy@0.5:', acc)

## Quick Diagnostic Plot

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(pred_prob[np.isfinite(pred_prob)], bins=20, alpha=0.8)
plt.title('PURITY Predicted Signal Probabilities')
plt.xlabel('pred_purity_signal')
plt.ylabel('count')
plt.show()